<h1>Chapter 5 - Text Clustering（文本聚类） and Topic Modeling（主题建模）</h1>
<i>Clustering documents using a wide variety of language models.</i>

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter05/Chapter%205%20-%20Text%20Clustering%20and%20Topic%20Modeling.ipynb)

---

This notebook is for Chapter 5 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---


In [ ]:
# %%capture
# !pip install bertopic datasets openai datamapplot

# **ArXiv Articles: Computation and Language**

In [ ]:
# Load data from huggingface
from datasets import load_dataset

"""
加载数据集
arxiv_nlp 这一数据集包含 1991 年至 2024 年间来自 ArXiv cs.CL（计算与语言）板块的 44 949 篇摘要
"""
dataset = load_dataset("maartengr/arxiv_nlp")["train"]  # arXiv 论文摘要数据集

# Extract metadata
abstracts = list(dataset["Abstracts"])
titles = list(dataset["Titles"])

# **A Common Pipeline for Text Clustering (文本聚类)**

## **1. Embedding Documents**

In [ ]:
from sentence_transformers import SentenceTransformer

# 我们将使用 MTEB 排行榜来选择嵌入模型
# Create an embedding for each abstract
embedding_model = SentenceTransformer('thenlper/gte-small')
# 为每个摘要创建嵌入向量
embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

In [ ]:
# Check the dimensions of the resulting embeddings
# (44949, 384)
embeddings.shape

## **2. Reducing the Dimensionality of Embeddings**

In [ ]:
from umap import UMAP

"""
UMAP 是一种降维算法
Uniform Manifold Approximation and Projection，统一流形逼近和投影

用于：
1.将高维的 BERT(ERT (Bidirectional Encoder Representations from Transformers)) 嵌入向量（如 768 维或 1536 维）降维到二维或三维
2.降维后的向量可以用于可视化（绘制散点图）
3.降维后的向量也可以作为 BERTopic 聚类的输入（默认使用 UMAP 先降到 5 维，再聚类

参数：
（1） n_components=5, 降维目标维度。将 384 维的嵌入向量降到 5 维。BERTopic 默认先降到 5 维再聚类，相比直接降到 2 维可视化，保留更多信息给聚类用
（2）min_dist=0.0, 点之间的最小距离。值越小，降维后点在低维空间中可以靠得越近，聚类更紧凑；值越大（如 0.5），点分布更分散。设为 0.0 最适合后续用 HDBSCAN 做密度聚类
（3）metric='cosine', 距离度量方式。使用余弦距离（cosine distance = 1 - cosine similarity），对文本嵌入向量的方向敏感，适合 Sentence-BERT 等生成的语义向量
（4）random_state=42, 随机种子。固定后每次运行结果一致，保证可复现

"""
# We reduce the input embeddings from 384 dimenions to 5 dimenions
umap_model = UMAP(
    n_components=5, min_dist=0.0, metric='cosine', random_state=42
)
reduced_embeddings = umap_model.fit_transform(embeddings)  # 降维后的嵌入向量

## **3. Cluster the Reduced Embeddings**

In [ ]:
from hdbscan import HDBSCAN

"""
HDBSCAN（Hierarchical Density-Based Spatial Clustering of Applications with Noise）
具有噪声的分层密度空间聚类

逐行解释：
1. HDBSCAN(...) — 实例化 HDBSCAN 聚类器。HDBSCAN 是一种【层次密度聚类】算法，能自动确定聚类数量，并将【噪声点标记为 -1】。

1.1 min_cluster_size=50 — 最小簇大小参数。
表示一个簇至少需要包含 50 个样本点。
值越大，生成的簇越少且越粗糙；值越小，越倾向于发现更细粒度的簇。

1.2 metric='euclidean' — 距离度量方式，使用欧氏距离计算样本间的相似度。
适用于经过降维后的稠密嵌入向量。

1.3 cluster_selection_method='eom' — 簇选择方法。
'eom'（Excess of Mass）倾向于选择更稳定、更持久的簇；另一种选项是 'leaf'，倾向于生成更多的簇。

2.fit(reduced_embeddings) — 对 降维后的嵌入向量 执行拟合操作。
reduced_embeddings 通常是对原始文本嵌入（如 Sentence-BERT 输出）经过 UMAP 或 PCA 降维得到的低维表示。【fit 会计算数据的密度层次结构并提取聚类。】

3.clusters = hdbscan_model.labels_ — 提取聚类标签结果。
labels_ 是一个一维数组，长度等于样本数，每个位置的整数值表示对应样本所属的簇编号，其中 【-1 表示该样本被识别为噪声/离群点，没有被分配到任何簇】

"""
# We fit the model and extract the clusters
hdbscan_model = HDBSCAN(
    min_cluster_size=50, metric='euclidean', cluster_selection_method='eom'
).fit(reduced_embeddings)
clusters = hdbscan_model.labels_

# How many clusters did we generate?
# 生成簇
# 156
len(set(clusters))

## **Inspecting the Clusters**

Manually inspect(手动检查簇) the first three documents in cluster 0:

In [ ]:
import numpy as np

# Print first three documents in cluster 0
# 打印簇0中的前三个文档
cluster = 0
"""
clusters == cluster — 比较每个文档所属的簇是否等于 0，返回布尔数组（True/False）
np.where(...) — 返回所有 True 位置的索引元组，格式为 (array([...],),)
[0] — 取出第一个（也是唯一一个）维度的一维索引数组
[:3] — 取前 3 个索引，即前 3 个属于簇 0 的文档位置
"""
for index in np.where(clusters == cluster)[0][:3]:
    # 根据索引取出对应文档的摘要文本的前300个字符
    print(abstracts[index][:300] + "... \n")

Next, we reduce our embeddings to 2-dimensions so that we can plot（可视化） them and get a rough understanding of the generated clusters.

In [ ]:
import pandas as pd

"""
可视化结果，这样就不用手动检查所有文档了
将降维后的 2 维嵌入向量（384维→2维）转为 DataFrame

"""
# Reduce 384-dimensional embeddings to 2 dimensions for easier visualization
reduced_embeddings = UMAP(
    n_components=2, min_dist=0.0, metric='cosine', random_state=42
).fit_transform(embeddings)

# Create dataframe
df = pd.DataFrame(reduced_embeddings, columns=["x", "y"])
# 新增 title 列，存入每篇文档的标题，用于后续可视化时标
df["title"] = titles

"""
新增 cluster 列，将每个样本的聚类标签（整数）转成字符串

为什么要转字符串？
因为聚类标签包含 -1（噪声点），转成字符串后，"-1" 与正常簇 "0", "1" 等区分开，方便按标签筛选和绘图（避免把 -1 当成数值参与计算）
"""
df["cluster"] = [str(c) for c in clusters]

# 选择离群点和非离群点（聚类）
# Select outliers and non-outliers (clusters)
# 筛选出非噪声点（簇标签不为 -1 的行），存到 clusters_df（用于正常簇的散点绘制
clusters_df = df.loc[df.cluster != "-1", :]

# 筛选出噪声点（簇标签为 -1 的行），存到 outliers_df（用于单独绘制为灰色散点）
outliers_df = df.loc[df.cluster == "-1", :]

### Static Plot(静态图)

In [ ]:
import matplotlib.pyplot as plt

# Plot outliers and non-outliers seperately
# 绘制非离群点
plt.scatter(outliers_df.x, outliers_df.y, alpha=0.05, s=2, c="grey")
# 绘制离群点
plt.scatter(
    clusters_df.x, clusters_df.y, c=clusters_df.cluster.astype(int),
    alpha=0.6, s=2, cmap='tab20b'
)
plt.axis('off')
# plt.savefig("matplotlib.png", dpi=300)  # Uncomment to save the graph as a .png

# From Text Clustering to Topic Modeling （主题建模）

## **BERTopic: A Modular Topic Modeling Framework**

In [ ]:
from bertopic import BERTopic

# 使用之前定义的模型训练我们的模型
# Train our model with our previously defined models
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
).fit(abstracts, embeddings)

Now, let's start exploring the topics that we got by running the code above.

In [ ]:
"""
| Topic | Count | Name | Representation |
| :--- | :--- | :--- | :--- |
| -1 | 14520 | -1_the_of_and_to | [the, of, and, to, in, we, that, language, for... |
| 0 | 2290 | 0_speech_asr_recognition_end | [speech, asr, recognition, end, acoustic, spea... |
| 1 | 1403 | 1_medical_clinical_biomedical_patient | [medical, clinical, biomedical, patient, healt... |
| 2 | 1156 | 2_sentiment_aspect_analysis_reviews | [sentiment, aspect, analysis, reviews, opinion... |
| 3 | 986 | 3_translation_nmt_machine_neural | [translation, nmt, machine, neural, bleu, engl... |
| ... | ... | ... | ... |
| 150 | 54 | 150_coherence_discourse_paragraph_text | [coherence, discourse, paragraph, text, cohesi... |
| 151 | 54 | 151_prompt_prompts_optimization_prompting | [prompt, prompts, optimization, prompting, llm... |
| 152 | 53 | 152_sentence_sts_embeddings_similarity | [sentence, sts, embeddings, similarity, embedd... |
| 153 | 53 | 153_counseling_mental_health_therapy | [counseling, mental, health, therapy, psychoth... |
| 154 | 50 | 154_backdoor_attacks_attack_triggers | [backdoor, attacks, attack, triggers, poisoned... |

说明：
（1）每个主题都由几个关键词表示，这些关键词在 Name 列中用“_”连接。
Name 列显示了最能代表该主题的四个关键词

（2）第一个主题被标记为 -1。该主题包含了所有无法归入某个主题的文档，这些文档被视为离群点


"""
topic_model.get_topic_info()


Hundreds of topics were generated using the default model! To get the top 10 keywords per topic as well as their c-TF-IDF weights, we can use the `get_topic()` function:

In [ ]:
"""
Top 10 关键词
基于这些关键词，该主题看起来是关于自动语音识别（automatic speech recognition，ASR）

[('speech', 0.028177697715245358),
 ('asr', 0.018971184497453525),
 ('recognition', 0.013457745472471012),
 ('end', 0.00980445092749381),
 ('acoustic', 0.009452082794507863),
  ('speaker', 0.0068822647060204885),
 ('audio', 0.006807649923681604),
 ('the', 0.0063343444687017645),
 ('error', 0.006320144717019838),
 ('automatic', 0.006290216996043161)]
"""
topic_model.get_topic(0)

We can use the `find_topics()` function to search for specific topics based on a search term. Let’s search for a topic about topic modeling:

In [ ]:
"""
于搜索词来查找特定主题

([22, -1, 1, 47, 32],
 [0.95456535, 0.91173744, 0.9074769, 0.9067007, 0.90510106])

这表明主题 22 与我们的搜索词具有较高的相似度（超过 0.95）
"""
topic_model.find_topics("topic modeling")

It returns that topic 22 has a relatively high similarity (0.95) with our search term. If we then inspect the topic, we can see that it is indeed a topic about topic modeling:

In [ ]:
"""
进一步查看该主题，可以看到它确实是一个关于主题建模的主题

[('topic', 0.06634619076655907),
 ('topics', 0.035308535091932707)
  ('lda', 0.016386314730705634),
 ('latent', 0.013372311924864435),
 ('document', 0.012973600191120576),
 ('documents', 0.012383715497143821),
 ('modeling', 0.011978375291037142),
 ('dirichlet', 0.010078277589545706),
 ('word', 0.008505619415413312),
 ('allocation', 0.007930890698168108)]
"""
topic_model.get_topic(22)

That seems like a topic that is, in part, characterized by the classic LDA technique. Let's see if the BERTopic paper was also assigned to topic 22:

In [ ]:
"""
确认下 BERTopic 文章的摘要是否也被分配到了这个主题
"""
# 22
topic_model.topics_[titles.index('BERTopic: Neural topic modeling with a class-based TF-IDF procedure')]

It is! We expected it might be because there are non-LDA specific words in the topic describtion such as "clustering" and "topic".

### **Visualizations（可视化）**

**Visualize Documents**

In [ ]:
# Visualize topics and documents
fig = topic_model.visualize_documents(
    titles,
    reduced_embeddings=reduced_embeddings,
    width=1200,
    hide_annotations=True
)

# Update fonts of legend for easier visualization
fig.update_layout(font=dict(size=16))

In [ ]:
# Visualize barchart with ranked keywords
# 可视化带有关键词排名的条形图
topic_model.visualize_barchart()

# Visualize relationships between topics
# 可视化主题之间的关系
topic_model.visualize_heatmap(n_clusters=30)

# Visualize the potential hierarchical structure of topics
# 可视化主题的潜在层次结构
topic_model.visualize_hierarchy()

## **Representation Models(表示模型，微调，重排器)**

In these examples that follow, we will update our topic representations **after** having trained our model. This allows for quick iteration. If, however, you want to use a representation model at the start of training, you will need to run it as follows:

```python
from bertopic.representation import KeyBERTInspired
from bertopic import BERTopic

# Create your representation model
representation_model = KeyBERTInspired()

# Use the representation model in BERTopic on top of the default pipeline
topic_model = BERTopic(representation_model=representation_model)
```

To use the representation models, we are first going to duplicate our topic model such that easily show the differences between a model with and without representation model.

In [ ]:
"""
在探索如何使用这些【表示模块】之前，我们需要做两件事。
我们要保存原始的主题表示，这样就更容易比较使用和不使用表示模型的结果
"""
# Save original representations
from copy import deepcopy
original_topics = deepcopy(topic_model.topic_representations_)

In [ ]:
"""
用于快速可视化主题词的差异
"""
def topic_differences(model, original_topics, nr_topics=5):
    """Show the differences in topic representations between two models """
    df = pd.DataFrame(columns=["Topic", "Original", "Updated"])
    for topic in range(nr_topics):

        # Extract top 5 words per topic per model
        og_words = " | ".join(list(zip(*original_topics[topic]))[0][:5])
        new_words = " | ".join(list(zip(*model.get_topic(topic)))[0][:5])
        df.loc[len(df)] = [topic, og_words, new_words]
    return df

### KeyBERTInspired

In [ ]:
from bertopic.representation import KeyBERTInspired

# 使用
# Update our topic representations using KeyBERTInspired
representation_model = KeyBERTInspired()
topic_model.update_topics(abstracts, representation_model=representation_model)

# Show topic differences
# 使用 c-TF-IDF 和前面展示的 KeyBERTInspired 技术，生成的主题表示中仍然存在显著的冗余。
"""
Topic	Original	Updated
0	speech | asr | recognition | end | acoustic	speech | encoder | phonetic | language | trans...
1	medical | clinical | biomedical | patient | he...	nlp | ehr | clinical | biomedical | language
2	sentiment | aspect | analysis | reviews | opinion	aspect | sentiment | aspects | sentiments | cl...
3	translation | nmt | machine | neural | bleu	translation | translating | translate | transl...
4	summarization | summaries | summary | abstract...	summarization | summarizers | summaries | summ...
"""
topic_differences(topic_model, original_topics)

### Maximal Marginal Relevance（最大边际相关性）

In [ ]:
from bertopic.representation import MaximalMarginalRelevance

# Update our topic representations to MaximalMarginalRelevance
representation_model = MaximalMarginalRelevance(diversity=0.5)
topic_model.update_topics(abstracts, representation_model=representation_model)

# Show topic differences
topic_differences(topic_model, original_topics)

## Text Generation（文本生成）



### Flan-T5

In [ ]:
from transformers import pipeline
from bertopic.representation import TextGeneration

"""
构建提示词
（1）筛选出的文档
（2）构成主题的关键词
"""
prompt = """I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: '[KEYWORDS]'.

Based on the documents and keywords, what is this topic about?"""

# Update our topic representations using Flan-T5
generator = pipeline('text2text-generation', model='google/flan-t5-small')
representation_model = TextGeneration(
    generator, prompt=prompt, doc_length=50, tokenizer="whitespace"
)
topic_model.update_topics(abstracts, representation_model=representation_model)

# Show topic differences
topic_differences(topic_model, original_topics)

### OpenAI

In [ ]:
import openai
from bertopic.representation import OpenAI

prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label in the following format:
topic: <short topic label>
"""

# Update our topic representations using GPT-3.5
client = openai.OpenAI(api_key="YOUR_KEY_HERE")
representation_model = OpenAI(
    client, model="gpt-3.5-turbo", exponential_backoff=True, chat=True, prompt=prompt
)
topic_model.update_topics(abstracts, representation_model=representation_model)

# Show topic differences
topic_differences(topic_model, original_topics)

In [ ]:
# Visualize topics and documents
# 可视化的前 20 个主题
fig = topic_model.visualize_document_datamap(
    titles,
    topics=list(range(20)),
    reduced_embeddings=reduced_embeddings,
    width=1200,
    label_font_size=11,
    label_wrap_width=20,
    use_medoids=True,
)
plt.savefig("datamapplot.png", dpi=300)


## **BONUS**: Word Cloud

Make sure to pip install `wordcloud` first in order to follow this bonus:


First, we need to make sure that each topic is described by a bit more words than just 10 as that would make for a much more interesting wordcloud.

In [ ]:
topic_model.update_topics(abstracts, top_n_words=500)

Then, we can run the following code to generate the wordcloud for our topic modeling topic:

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

def create_wordcloud(model, topic):
    plt.figure(figsize=(10,5))
    text = {word: value for word, value in model.get_topic(topic)}
    wc = WordCloud(background_color="white", max_words=1000, width=1600, height=800)
    wc.generate_from_frequencies(text)
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.show()

# Show wordcloud
create_wordcloud(topic_model, topic=17)